# Fact API Metrics

## Overview
Creates **fact_api_metrics** fact table containing comprehensive API performance and system metrics.

**Source**: api_data (silver)
**Dimensions**: dim_device, dim_time
**Target**: telecom_catalog.gold_schema.fact_api_metrics
**Partition**: event_date

---

## Step 1: Create Fact Table
Define partitioned fact table with device_key, time_key (FKs), 23 API metrics, and health status fields.

In [0]:
%sql
CREATE OR REPLACE TABLE telecom_catalog.gold_schema.fact_api_metrics
(
    device_key BIGINT NOT NULL,
    time_key INT NOT NULL,

    active_sessions BIGINT,
    anomaly_score DOUBLE,
    cache_hit_ratio DOUBLE,
    crash_count BIGINT,
    disk_read_ops BIGINT,
    disk_write_ops BIGINT,
    failed_requests BIGINT,
    fan_speed_rpm BIGINT,
    io_wait_time_ms BIGINT,
    latency_ms_p99 BIGINT,
    packet_loss_percentage DOUBLE,
    power_usage_watts BIGINT,
    process_count BIGINT,
    queue_length BIGINT,
    reboot_count BIGINT,
    request_count BIGINT,
    service_restart_count BIGINT,
    tcp_connections BIGINT,
    temperature_celsius DOUBLE,
    thread_count BIGINT,
    unauthorized_access_attempts BIGINT,
    uptime_percentage DOUBLE,

    temperature_status STRING,
    network_health STRING,
    uptime_category STRING,

    event_date DATE
    )
USING DELTA
PARTITIONED BY (event_date);

---
## Step 2: Prepare Source View
Join api_data with dim_device (on api_device_id) and dim_time (on event_date).

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_fact_api_source AS

SELECT

    dd.device_key,
    dt.time_key,

    api.active_sessions,
    api.anomaly_score,
    api.cache_hit_ratio,
    api.crash_count,
    api.disk_read_ops,
    api.disk_write_ops,
    api.failed_requests,
    api.fan_speed_rpm,
    api.io_wait_time_ms,
    api.latency_ms_p99,
    api.packet_loss_percentage,
    api.power_usage_watts,
    api.process_count,
    api.queue_length,
    api.reboot_count,
    api.request_count,
    api.service_restart_count,
    api.tcp_connections,
    api.temperature_celsius,
    api.thread_count,
    api.unauthorized_access_attempts,
    api.uptime_percentage,

    api.temperature_status,
    api.network_health,
    api.uptime_category,

    api.event_date

FROM telecom_catalog.silver_schema.api_data api

INNER JOIN telecom_catalog.gold_schema.dim_device dd
    ON api.device_id = dd.api_device_id

INNER JOIN telecom_catalog.gold_schema.dim_time dt
    ON api.event_date = dt.full_date;

---
## Step 3: Validate Source Count
Verify source view record count.

In [0]:
%sql
SELECT COUNT(*)
FROM vw_fact_api_source;

COUNT(*)
1000000


---
## Step 4: Check Foreign Keys
Validate no NULL values in device_key or time_key.

In [0]:
%sql
SELECT
    SUM(CASE WHEN device_key IS NULL THEN 1 ELSE 0 END) AS null_device_key,
        SUM(CASE WHEN time_key IS NULL THEN 1 ELSE 0 END) AS null_time_key
            FROM vw_fact_api_source;

null_device_key,null_time_key
0,0


---
## Step 5: Load Fact Table
Insert all records from source view into partitioned fact table.

In [0]:
%sql
INSERT INTO telecom_catalog.gold_schema.fact_api_metrics

SELECT
    device_key,
    time_key,

    active_sessions,
    anomaly_score,
    cache_hit_ratio,
    crash_count,
    disk_read_ops,
    disk_write_ops,
    failed_requests,
    fan_speed_rpm,
    io_wait_time_ms,
    latency_ms_p99,
    packet_loss_percentage,
    power_usage_watts,
    process_count,
    queue_length,
    reboot_count,
    request_count,
    service_restart_count,
    tcp_connections,
    temperature_celsius,
    thread_count,
    unauthorized_access_attempts,
    uptime_percentage,

    temperature_status,
    network_health,
    uptime_category,

    event_date

FROM vw_fact_api_source;

num_affected_rows,num_inserted_rows
1000000,1000000


---
## Step 6: Validate Load
Verify final record count after insert.

In [0]:
%sql
SELECT COUNT(*)
FROM telecom_catalog.gold_schema.fact_api_metrics;

COUNT(*)
1000000


---
## Step 7: Check Partitions
Verify records per event_date partition.

In [0]:
%sql
SELECT
    event_date,
        COUNT(*) AS records
        FROM telecom_catalog.gold_schema.fact_api_metrics
        GROUP BY event_date
        ORDER BY event_date;

event_date,records
2026-06-14,45254
2026-06-15,86400
2026-06-16,86400
2026-06-17,86400
2026-06-18,86400
2026-06-19,86400
2026-06-20,86400
2026-06-21,86400
2026-06-22,86400
2026-06-23,86400


---
## Step 8: Optimize Table
Compact small files for better query performance.

In [0]:
%sql
OPTIMIZE telecom_catalog.gold_schema.fact_api_metrics;

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 13, null, null, 0, 0, 13, 13, true, 0, 0, 1787801621995, 1787801623720, 8, 0, null, List(0, 0), null, 28, 28, 0, 0, null, null, 0)"


---
## Step 9: Compute Statistics
Update table statistics for query optimization.

In [0]:
%sql
ANALYZE TABLE telecom_catalog.gold_schema.fact_api_metrics
COMPUTE STATISTICS;